In [1]:
import torch
import random
import tqdm

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
from joblib import Parallel, delayed
from multiprocessing import Pool
from pymfe.mfe import MFE
from tabicl_prior import sample_datasets

In [4]:
N_DATASETS = 1000
SEED = 42
N_JOBS = 8

In [5]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [6]:
device = 'cuda' if torch.accelerator.is_available() else 'cpu'
device

'cuda'

In [7]:
datasets = sample_datasets('clf', stage=1, n_datasets=N_DATASETS, seed=SEED, n_jobs=N_JOBS)

In [8]:
def extract(args):
    i, X, y = args
    mfe = MFE(groups='complexity', features='all', summary='mean', random_state=SEED)
    mfe.fit(X, y, suppress_warnings=True)
    measures, values = mfe.extract(suppress_warnings=True)
    return {'id': i, 'n_classes': len(np.unique(y)), 'n_rows': X.shape[0], 'n_cols': X.shape[1], **dict(zip(measures, map(float, values)))}

In [ ]:
complexity = []
for i, data in enumerate(datasets):
    X = np.concatenate([data['X_train'], data['X_test']], axis=0)
    y = np.concatenate([data['y_train'], data['y_test']], axis=0)
    complexity.append(extract((i, X, y)))   

In [ ]:
df = pd.DataFrame(complexity)
df

In [ ]:
df.to_csv('complexity.csv')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 2.8))
for ax, col in zip(axes, df.columns[1:4]):
    if col not in df:
        ax.set_visible(False); continue
    v = df[col]
    if v.nunique() <= 30:                                   # discreto: uma barra por valor
        cont = v.value_counts().sort_index()
        ax.bar(cont.index, cont.values, width=0.8)
    else:
        ax.hist(v, bins=30, linewidth=1)
    ax.set_title(col)
    ax.grid(axis="x", visible=False)
fig.suptitle("Características dos datasets - eixo y: número de datasets", x=0.01, ha="left")
fig.tight_layout(); plt.show()

In [ ]:
meta = ["id", "n_classes", "n_rows", "n_cols"]
medidas = df.drop(columns=meta).dropna(axis="columns", how="all")
variaveis = medidas.median().sort_values().index.tolist()

fig, ax = plt.subplots(figsize=(max(8, 0.55 * len(variaveis)), 4))
ax.boxplot([medidas[c].dropna() for c in variaveis])
ax.set_xticks(
    range(1, len(variaveis) + 1), variaveis,
    rotation=60, ha="right", rotation_mode="anchor",
)
ax.set_ylabel("valor da medida")
ax.set_title(f"Medidas de complexidade · n = {len(df)} datasets")
ax.grid(axis="x", visible=False)
fig.tight_layout(); plt.show()

In [ ]:
cols_corr = df.iloc[:, 4:].dropna(axis="columns", how="all").columns
corr = df[cols_corr].corr(method="spearman")
n = len(cols_corr)
fig, ax = plt.subplots(figsize=(0.45 * n + 2.5, 0.45 * n + 1))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)     # azul = negativa, vermelho = positiva
ax.set_xticks(range(n), cols_corr, rotation=90)
ax.set_yticks(range(n), cols_corr)
ax.grid(False)
for i in range(n):
    for j in range(n):
        r = corr.iloc[i, j]
        if i != j and abs(r) >= 0.7:                          # anota só as correlações fortes
            ax.text(j, i, f"{r:.1f}", ha="center", va="center", fontsize=6,
                    color="white" if abs(r) > 0.75 else "black")
fig.colorbar(im, ax=ax, fraction=0.04, label="correlação de Spearman")
ax.set_title("Correlação entre medidas de complexidade (números: |ρ| ≥ 0.7)")
fig.tight_layout(); plt.show()

In [ ]:
# Próximos passos
# - rodar na axia
# - usar o prior do tabicl v2
# - definir os hyperparametros de geração das bases
# - rodar as medias de complexidade para bases reais
# - vamos gerar dados siteticos?

# Mais comentarios
# - mais meta-features como impureza de gini (arvore)
# - usar classificadores como metricas
# - 